In [1]:
"""
LPG demand calibration and reseller client allocation (Nigeria, 1 km, EPSG:3857)

This notebook turns the preferred-reseller output from the previous step into:
- a final LPG use-share raster per pixel
- reseller-level demand metrics in a new GeoPackage

Core assumptions
-----------------
- walking access is valid only up to WALK_THRESHOLD_MIN minutes
- motorized access is valid only up to CAR_THRESHOLD_MIN minutes
- outside the access threshold, LPG use is zero
- within each group (urban/rural), one calibration factor scales eligible demand
- urban and rural target shares are matched on the processed pixel domain
- national share is the implied weighted result of urban/rural targets

Outputs
-------
Main output raster:
- lpg_use_share.tif

Main vector output:
- sell_point_clients.gpkg with clients, clients_walk, clients_car,
  clients_max_ideal, clients_max_ideal_walk, clients_max_ideal_car
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling

DATA_DIR = "dataset_big"

# Inputs
PIXEL_PREFERENCE_GPKG = f"{DATA_DIR}/huff_preferred_distributor_per_pixel.gpkg"
PIXEL_PREFERENCE_LAYER = "pixel_preference"
RESELLER_GPKG = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"
URBAN_RASTER = f"{DATA_DIR}/Urban.tif"
BOUNDARY_FILE = f"{DATA_DIR}/Country_boundaries.geojson"

# Outputs
OUTPUT_LPG_USE_RASTER = f"{DATA_DIR}/lpg_use_share.tif"
OUTPUT_RESELLER_GPKG = f"{DATA_DIR}/sell_point_clients.gpkg"
OUTPUT_RESELLER_LAYER = "sell_point_clients"

# ID convention from step 3/4.2
RESELLER_ID_COLUMN = "id_res&fil"

# Calibration parameters
WALK_THRESHOLD_MIN = 30
CAR_THRESHOLD_MIN = 45
urban_share = 0.25468
rural_share = 0.027905
URBAN_THRESHOLD = 20  # verify value 20

# Nodata conventions
NODATA_FLOAT = -9999.0
NODATA_INT = -1

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def _read_raster(path: str):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile, nodata


def _align_to_reference(path: str, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    with rasterio.open(path) as src:
        dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _safe_numeric(values) -> np.ndarray:
    return pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=np.float64)


def _read_reseller_ids(gdf: gpd.GeoDataFrame) -> np.ndarray:
    if RESELLER_ID_COLUMN not in gdf.columns:
        raise KeyError(f"Missing column '{RESELLER_ID_COLUMN}' in reseller layer.")

    rid = pd.to_numeric(gdf[RESELLER_ID_COLUMN], errors="coerce")
    if rid.isna().any():
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' contains non-numeric values.")
    if (rid <= 0).any():
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must contain positive values.")
    if not rid.is_unique:
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must be unique in layer '{RESELLER_LAYER}'.")

    return rid.astype(np.int64).to_numpy()


def _write_float_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype(np.float32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


# ---------------------------------------------------------------------
# LOAD INPUTS
# ---------------------------------------------------------------------
print("[1/7] Loading pixel preference table...")
pixel_df = gpd.read_file(PIXEL_PREFERENCE_GPKG, layer=PIXEL_PREFERENCE_LAYER)
if pixel_df.empty:
    raise RuntimeError("Pixel preference layer is empty.")

required_columns = {
    "row",
    "col",
    "walk_share",
    "car_share",
    "best_reseller_id_walk",
    "best_time_walk_min",
    "best_reseller_id_car",
    "best_time_car_min",
}
missing_columns = required_columns.difference(pixel_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns in pixel preference layer: {sorted(missing_columns)}")

print(f"Pixels loaded: {len(pixel_df):,}")

print("[2/7] Loading population and urban rasters...")
pop, pop_profile, pop_nodata = _read_raster(POPULATION_RASTER)
transform = pop_profile["transform"]
crs = pop_profile["crs"]
height, width = pop.shape

urban = _align_to_reference(URBAN_RASTER, pop_profile, Resampling.nearest)
urban = np.where(np.isfinite(urban), urban, np.nan).astype(np.float32)

print(f"Grid: {width} x {height}, CRS={crs}")

print("[3/7] Building Nigeria mask...")
boundary = gpd.read_file(BOUNDARY_FILE)
if boundary.crs != crs:
    boundary = boundary.to_crs(crs)
nigeria_geom = boundary.geometry.union_all()
mask_nigeria = geometry_mask(
    [nigeria_geom],
    transform=transform,
    invert=True,
    out_shape=(height, width),
)

pop = np.where(np.isfinite(pop), np.maximum(pop, 0.0), np.nan).astype(np.float32)

# ---------------------------------------------------------------------
# PREPARE PIXEL-LEVEL CALIBRATION DATA
# ---------------------------------------------------------------------
print("[4/7] Preparing access eligibility and calibration factors...")

rows = pd.to_numeric(pixel_df["row"], errors="coerce").to_numpy(dtype=np.int64)
cols = pd.to_numeric(pixel_df["col"], errors="coerce").to_numpy(dtype=np.int64)
inside_grid = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
if not np.all(inside_grid):
    pixel_df = pixel_df.loc[inside_grid].copy()
    rows = rows[inside_grid]
    cols = cols[inside_grid]

pixel_pop = pop[rows, cols].astype(np.float64)
valid_pop = np.isfinite(pixel_pop) & (pixel_pop > 0)
if not np.all(valid_pop):
    pixel_df = pixel_df.loc[valid_pop].copy()
    rows = rows[valid_pop]
    cols = cols[valid_pop]
    pixel_pop = pixel_pop[valid_pop]

walk_share = np.clip(_safe_numeric(pixel_df["walk_share"]), 0.0, 1.0)
car_share = np.clip(_safe_numeric(pixel_df["car_share"]), 0.0, 1.0)
share_sum = walk_share + car_share
if np.any(share_sum <= 0):
    raise RuntimeError("Invalid walk/car shares: at least one pixel has zero total share.")
walk_share = walk_share / share_sum
car_share = car_share / share_sum

walk_time = _safe_numeric(pixel_df["best_time_walk_min"])
car_time = _safe_numeric(pixel_df["best_time_car_min"])
walk_eligible = np.isfinite(walk_time) & (walk_time <= WALK_THRESHOLD_MIN)
car_eligible = np.isfinite(car_time) & (car_time <= CAR_THRESHOLD_MIN)

urban_pixel = urban[rows, cols]
urban_flag = np.where(np.isfinite(urban_pixel), urban_pixel >= URBAN_THRESHOLD, False)
rural_flag = ~urban_flag

pixel_df = pixel_df.copy()
pixel_df["row"] = rows
pixel_df["col"] = cols

urban_pop_total = float(np.nansum(pixel_pop[urban_flag]))
rural_pop_total = float(np.nansum(pixel_pop[rural_flag]))
if urban_pop_total <= 0 or rural_pop_total <= 0:
    raise RuntimeError("Urban or rural population total is zero. Check the urban raster and population alignment.")

urban_eligible_pop = float(np.nansum(pixel_pop[urban_flag] * (walk_share[urban_flag] * walk_eligible[urban_flag] + car_share[urban_flag] * car_eligible[urban_flag])))
rural_eligible_pop = float(np.nansum(pixel_pop[rural_flag] * (walk_share[rural_flag] * walk_eligible[rural_flag] + car_share[rural_flag] * car_eligible[rural_flag])))

if urban_eligible_pop <= 0:
    raise RuntimeError("Urban eligible population is zero. The 30-minute access threshold is too strict or the inputs are inconsistent.")
if rural_eligible_pop <= 0:
    raise RuntimeError("Rural eligible population is zero. The thresholds are too strict or the inputs are inconsistent.")

urban_factor = (urban_share * urban_pop_total) / urban_eligible_pop
rural_factor = (rural_share * rural_pop_total) / rural_eligible_pop

if urban_factor > 1.0 + 1e-9:
    raise RuntimeError(f"Urban calibration factor is above 1 ({urban_factor:.4f}). The calibration is infeasible with the current hard cutoff.")
if rural_factor > 1.0 + 1e-9:
    raise RuntimeError(f"Rural calibration factor is above 1 ({rural_factor:.4f}). The calibration is infeasible with the current hard cutoff.")

eligible_weight = walk_share * walk_eligible.astype(np.float64) + car_share * car_eligible.astype(np.float64)
region_factor = np.where(urban_flag, urban_factor, rural_factor)
lpg_use_share_pixel = region_factor * eligible_weight
lpg_use_share_pixel = np.clip(lpg_use_share_pixel, 0.0, 1.0)

clients_walk_pixel = pixel_pop * walk_share * walk_eligible.astype(np.float64) * region_factor
clients_car_pixel = pixel_pop * car_share * car_eligible.astype(np.float64) * region_factor
clients_pixel = clients_walk_pixel + clients_car_pixel
clients_max_ideal_walk_pixel = pixel_pop * walk_share
clients_max_ideal_car_pixel = pixel_pop * car_share

national_pop = float(np.nansum(pixel_pop))
national_eligible_pop = float(np.nansum(pixel_pop * eligible_weight))
national_clients = float(np.nansum(clients_pixel))
expected_national_clients = float(urban_share * urban_pop_total + rural_share * rural_pop_total)

print("\n=== Calibration factors ===")
print(f"Urban factor: {urban_factor:.6f}")
print(f"Rural factor: {rural_factor:.6f}")
print("\n=== Pre-output checks ===")
print(f"National population: {national_pop:,.0f}")
print(f"National eligible population: {national_eligible_pop:,.0f}")
print(f"Expected national LPG users: {expected_national_clients:,.0f}")
print(f"Actual national LPG users   : {national_clients:,.0f}")
print(f"Difference                  : {national_clients - expected_national_clients:+,.6f}")

# ---------------------------------------------------------------------
# BUILD LPG USE SHARE RASTER
# ---------------------------------------------------------------------
print("[5/7] Writing LPG use-share raster...")

lpg_use_share_raster = np.zeros((height, width), dtype=np.float32)
lpg_use_share_raster[~mask_nigeria] = NODATA_FLOAT
lpg_use_share_raster[rows, cols] = lpg_use_share_pixel.astype(np.float32)
_write_float_raster(OUTPUT_LPG_USE_RASTER, lpg_use_share_raster, pop_profile)
print(f"Saved raster: {OUTPUT_LPG_USE_RASTER}")

# ---------------------------------------------------------------------
# AGGREGATE CLIENTS PER RESELLER
# ---------------------------------------------------------------------
print("[6/7] Aggregating clients by reseller...")

reseller_df = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
if reseller_df.empty:
    raise RuntimeError("Reseller layer is empty.")
if reseller_df.crs != crs:
    reseller_df = reseller_df.to_crs(crs)
reseller_df = reseller_df[reseller_df.geometry.notna()].copy()
reseller_df = reseller_df[reseller_df.geometry.geom_type == "Point"].copy()
if reseller_df.empty:
    raise RuntimeError("No point geometries found in the reseller layer.")

reseller_id = _read_reseller_ids(reseller_df)
reseller_df = reseller_df.copy()
reseller_df["reseller_id"] = reseller_id.astype(np.int64)

walk_reseller_id = pd.to_numeric(pixel_df["best_reseller_id_walk"], errors="coerce").fillna(NODATA_INT).astype(np.int64).to_numpy()
car_reseller_id = pd.to_numeric(pixel_df["best_reseller_id_car"], errors="coerce").fillna(NODATA_INT).astype(np.int64).to_numpy()

pixel_client_table = pd.DataFrame(
    {
        "walk_reseller_id": walk_reseller_id,
        "car_reseller_id": car_reseller_id,
        "clients_walk": clients_walk_pixel,
        "clients_car": clients_car_pixel,
        "clients": clients_pixel,
        "clients_max_ideal_walk": clients_max_ideal_walk_pixel,
        "clients_max_ideal_car": clients_max_ideal_car_pixel,
        "pixel_population": pixel_pop,
    }
)

walk_clients = (
    pixel_client_table.loc[pixel_client_table["walk_reseller_id"] >= 0]
    .groupby("walk_reseller_id", as_index=False)
    .agg(
        clients_walk=("clients_walk", "sum"),
        clients_max_ideal_walk=("clients_max_ideal_walk", "sum"),
    )
    .rename(columns={"walk_reseller_id": "reseller_id"})
)

car_clients = (
    pixel_client_table.loc[pixel_client_table["car_reseller_id"] >= 0]
    .groupby("car_reseller_id", as_index=False)
    .agg(
        clients_car=("clients_car", "sum"),
        clients_max_ideal_car=("clients_max_ideal_car", "sum"),
    )
    .rename(columns={"car_reseller_id": "reseller_id"})
)

clients_agg = walk_clients.merge(car_clients, on="reseller_id", how="outer")
clients_agg["clients_walk"] = clients_agg["clients_walk"].fillna(0.0)
clients_agg["clients_car"] = clients_agg["clients_car"].fillna(0.0)
clients_agg["clients"] = clients_agg["clients_walk"] + clients_agg["clients_car"]
clients_agg["clients_max_ideal_walk"] = clients_agg["clients_max_ideal_walk"].fillna(0.0)
clients_agg["clients_max_ideal_car"] = clients_agg["clients_max_ideal_car"].fillna(0.0)
clients_agg["clients_max_ideal"] = clients_agg["clients_max_ideal_walk"] + clients_agg["clients_max_ideal_car"]

reseller_out = reseller_df.merge(clients_agg, on="reseller_id", how="left")
for col in ["clients_walk", "clients_car", "clients", "clients_max_ideal_walk", "clients_max_ideal_car", "clients_max_ideal"]:
    reseller_out[col] = reseller_out[col].fillna(0.0).astype(np.float64)

# Keep the attribute table readable and stable.
reseller_out = gpd.GeoDataFrame(reseller_out, geometry="geometry", crs=crs)
reseller_out.to_file(OUTPUT_RESELLER_GPKG, layer=OUTPUT_RESELLER_LAYER, driver="GPKG")
print(f"Saved reseller clients table: {OUTPUT_RESELLER_GPKG} | layer={OUTPUT_RESELLER_LAYER}")
print(f"Reseller id column: {RESELLER_ID_COLUMN}")

# ---------------------------------------------------------------------
# MAIN SUMMARY
# ---------------------------------------------------------------------
print("\n=== MAIN SUMMARY ===")
print(f"Urban population total  : {urban_pop_total:,.0f}")
print(f"Urban eligible population: {urban_eligible_pop:,.0f}")
print(f"Urban target share      : {urban_share:.6f}")
print(f"Urban achieved share    : {national_clients * 0.0 + (urban_factor * urban_eligible_pop / urban_pop_total):.6f}")
print(f"Urban factor applied    : {urban_factor:.6f}")
print(f"Rural population total  : {rural_pop_total:,.0f}")
print(f"Rural eligible population: {rural_eligible_pop:,.0f}")
print(f"Rural target share      : {rural_share:.6f}")
print(f"Rural achieved share    : {national_clients * 0.0 + (rural_factor * rural_eligible_pop / rural_pop_total):.6f}")
print(f"Rural factor applied    : {rural_factor:.6f}")
print(f"National actual share    : {national_clients / national_pop:.6f}")
print(f"National target share    : {expected_national_clients / national_pop:.6f}")
print("\nDone.")


[1/7] Loading pixel preference table...
Pixels loaded: 563,482
[2/7] Loading population and urban rasters...
Grid: 1337 x 1085, CRS=EPSG:3857
[3/7] Building Nigeria mask...
[4/7] Preparing access eligibility and calibration factors...

=== Calibration factors ===
Urban factor: 0.528938
Rural factor: 0.614922

=== Pre-output checks ===
National population: 205,003,117
National eligible population: 49,133,459
Expected national LPG users: 26,432,110
Actual national LPG users   : 26,432,110
Difference                  : +0.000000
[5/7] Writing LPG use-share raster...
Saved raster: dataset_big/lpg_use_share.tif
[6/7] Aggregating clients by reseller...
Saved reseller clients table: dataset_big/sell_point_clients.gpkg | layer=sell_point_clients
Reseller id column: id_res&fil

=== MAIN SUMMARY ===
Urban population total  : 91,330,605
Urban eligible population: 43,975,030
Urban target share      : 0.254680
Urban achieved share    : 0.254680
Urban factor applied    : 0.528938
Rural population to

In [2]:
# Diagnostics and final calibration checks
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.warp import Resampling

# Reuse the objects created in the previous cell when the notebook is run in order.

with rasterio.open(OUTPUT_LPG_USE_RASTER) as src:
    lpg_use = src.read(1).astype(np.float32)
    lpg_nodata = src.nodata
if lpg_nodata is not None:
    lpg_use = np.where(lpg_use == lpg_nodata, np.nan, lpg_use)

pop, pop_profile, _ = _read_raster(POPULATION_RASTER)
urban = _align_to_reference(URBAN_RASTER, pop_profile, Resampling.nearest)
urban = np.where(np.isfinite(urban), urban, np.nan).astype(np.float32)
transform = pop_profile["transform"]
crs = pop_profile["crs"]
height, width = pop.shape

boundary = gpd.read_file(BOUNDARY_FILE)
if boundary.crs != crs:
    boundary = boundary.to_crs(crs)
nigeria_geom = boundary.geometry.union_all()
mask_nigeria = geometry_mask(
    [nigeria_geom],
    transform=transform,
    invert=True,
    out_shape=(height, width),
)

pop = np.where(np.isfinite(pop), np.maximum(pop, 0.0), np.nan).astype(np.float32)
urban_mask = mask_nigeria & np.isfinite(urban) & (urban >= URBAN_THRESHOLD)
rural_mask = mask_nigeria & ~urban_mask

urban_pop = float(np.nansum(pop[urban_mask]))
rural_pop = float(np.nansum(pop[rural_mask]))
national_pop = float(np.nansum(pop[mask_nigeria]))

urban_clients = float(np.nansum(pop[urban_mask] * lpg_use[urban_mask]))
rural_clients = float(np.nansum(pop[rural_mask] * lpg_use[rural_mask]))
national_clients = float(np.nansum(pop[mask_nigeria] * lpg_use[mask_nigeria]))

urban_actual = urban_clients / urban_pop if urban_pop > 0 else np.nan
rural_actual = rural_clients / rural_pop if rural_pop > 0 else np.nan
national_actual = national_clients / national_pop if national_pop > 0 else np.nan

target_national = (urban_share * urban_pop + rural_share * rural_pop) / national_pop if national_pop > 0 else np.nan

check_table = pd.DataFrame(
    {
        "group": ["urban", "rural", "national"],
        "target_share": [urban_share, rural_share, target_national],
        "actual_share": [urban_actual, rural_actual, national_actual],
    }
)
check_table["difference"] = check_table["actual_share"] - check_table["target_share"]

print("=== CALIBRATION CHECK ===")
print(check_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Check reseller aggregation
reseller_clients = gpd.read_file(OUTPUT_RESELLER_GPKG, layer=OUTPUT_RESELLER_LAYER)
for col in ["clients_walk", "clients_car", "clients", "clients_max_ideal"]:
    reseller_clients[col] = pd.to_numeric(reseller_clients[col], errors="coerce").fillna(0.0)

print("\n=== RESELLER SUMMARY ===")
print(f"Resellers in output            : {len(reseller_clients):,}")
print(f"Resellers with clients > 0     : {int((reseller_clients['clients'] > 0).sum()):,}")
print(f"National clients from raster   : {national_clients:,.0f}")
print(f"National clients from resellers: {float(reseller_clients['clients'].sum()):,.0f}")
print(f"Difference                     : {float(reseller_clients['clients'].sum() - national_clients):+.6f}")

print("\nTop 10 resellers by actual clients")
cols_to_show = [
    c for c in ["reseller_id", "id_res&fil", "id_depot", "clients", "clients_walk", "clients_car", "clients_max_ideal"]
    if c in reseller_clients.columns
]
print(
    reseller_clients.sort_values("clients", ascending=False)[cols_to_show]
    .head(10)
    .to_string(index=False)
)

print("\nTop 10 resellers by ideal assigned population")
print(
    reseller_clients.sort_values("clients_max_ideal", ascending=False)[cols_to_show]
    .head(10)
    .to_string(index=False)
)


=== CALIBRATION CHECK ===
   group  target_share  actual_share  difference
   urban      0.254680      0.254122   -0.000558
   rural      0.027905      0.027847   -0.000058
national      0.128941      0.128660   -0.000281

=== RESELLER SUMMARY ===
Resellers in output            : 2,792
Resellers with clients > 0     : 2,184
National clients from raster   : 26,432,108
National clients from resellers: 26,432,110
Difference                     : +1.988044

Top 10 resellers by actual clients
 reseller_id  id_res&fil  id_depot       clients  clients_walk   clients_car  clients_max_ideal
         653         653       NaN 231625.328081 220622.152979  11003.175103      464140.210513
         601         601       NaN 148982.169899 135170.915912  13811.253986      351305.716452
        2194        2194       NaN 135958.233759 134175.340762   1782.892997      257039.863115
         621         621       NaN 134234.693451 100802.824970  33431.868481      290405.675410
          11          11   